# QAOA-GPT inference demo

In [1]:
import pandas as pd
import networkx as nx
import random
from tqdm import tqdm

from src.model_interface import QAOA_GPT
from src.adapt_utils import compute_metrics 


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## Loading a model

In [2]:
qaoa_gpt = QAOA_GPT(
    model_ckpt='nanoGPT/out-10_nodes_gnn/gpt_ckpt_3500_gnn_ar_0_95553__er_0_0.pt',
    data_dir='nanoGPT/data/10_nodes_gnn', # to take meta.pkl file 
)


Model type: gpt
Pool type: qaoa_double_pool
Embedding method: gnn
Number of nodes: 10
Initiating nanoGPT model with padding support
number of parameters: 11.68M


In [3]:
# qaoa_gpt = QAOA_GPT(
#     model_ckpt='nanoGPT/models/n10w_qaoa_mixer/ckpt_16000_gemb__ar_0_96584__er_0_0.pt',
#     data_dir='nanoGPT/models/n10w_qaoa_mixer/data', # to take meta.pkl file 
# )

## Generating random graphs

In [4]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [5]:
tqdm.pandas()

Modify this nodes here to match the model before run

In [6]:
n_graphs = 5
n_nodes = 10

In [7]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [8]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 1, {'weight': 0.12}), (0, 2, {'weight': 0.07}), (0, 3, {'weight': 0.7}), (0, 4, {'weight': 0.71}), (0, 6, {'weight': 0.48}), (0, 7, {'weight': 0.89}), (0, 8, {'weight': 0.61}), (1, 2, {'weight': 0.49}), (1, 4, {'weight': 0.45}), (1, 5, {'weight': 0.62}), (1, 6, {'weight': 0.71}), (1, 8, {'weight': 0.66}), (2, 5, {'weight': 0.88}), (2, 6, {'weight': 0.26}), (3, 5, {'weight': 0.21}), (3, 6, {'weight': 0.34}), (3, 7, {'weight': 0.4}), (3, 8, {'weight': 0.86}), (3, 9, {'weight': 0.03}), (4, 6, {'weight': 0.76}), (4, 7, {'weight': 0.27}), (4, 9, {'weight': 0.49}), (5, 6, {'weight': 0.3}), (6, 7, {'weight': 0.93}), (6, 8, {'weight': 0.1}), (6, 9, {'weight': 0.43}), (7, 8, {'weight': 0.44})])

## Generate circuits with QAOA-GPT model

In [9]:
embedding_method = qaoa_gpt.embedding_method
print(f"Using embedding method: {embedding_method}")

qaoa_gpt_circ_df = qaoa_gpt.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    # calculate_classic_maxcut=True, # to create col enery_gurobi. Default:True
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
)

Using embedding method: gnn


Preparing graphs...:   0%|          | 0/5 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2027-11-29


Preparing graphs...: 100%|██████████| 5/5 [00:00<00:00, 150.60it/s]


Performing gnn embedding
GNN model initialized on device: cuda


GNN:   0%|          | 0/5 [00:00<?, ?it/s]/home/mrzaizai2k/anaconda3/envs/adapt_gpt/lib/python3.10/site-packages/torch_geometric/utils/convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  data_dict[key] = torch.as_tensor(value)
GNN: 100%|██████████| 5/5 [00:00<00:00, 32.70it/s]


GNN shape: (5, 500)


Inference. Current batch: n_edges: 39, n_graphs: 1: 100%|██████████| 5/5 [00:02<00:00,  1.83it/s]


In [10]:
sample_gr = graphs_edgelist_list_dict['er_graph_0'].edges(data=True)
sample_gr

EdgeDataView([(0, 1, {'weight': 0.02}), (0, 2, {'weight': 0.6}), (0, 3, {'weight': 0.88}), (0, 4, {'weight': 0.76}), (0, 5, {'weight': 0.33}), (0, 6, {'weight': 0.02}), (0, 7, {'weight': 0.47}), (0, 8, {'weight': 0.86}), (0, 9, {'weight': 0.97}), (1, 2, {'weight': 0.38}), (1, 3, {'weight': 0.03}), (1, 4, {'weight': 0.71}), (1, 6, {'weight': 0.98}), (1, 7, {'weight': 0.65}), (1, 8, {'weight': 0.82}), (1, 9, {'weight': 0.35}), (2, 3, {'weight': 0.74}), (2, 4, {'weight': 0.42}), (2, 5, {'weight': 0.4}), (2, 6, {'weight': 0.1}), (2, 7, {'weight': 0.97}), (2, 8, {'weight': 0.28}), (2, 9, {'weight': 0.5}), (3, 4, {'weight': 0.53}), (3, 5, {'weight': 0.58}), (3, 6, {'weight': 0.85}), (3, 7, {'weight': 0.84}), (3, 9, {'weight': 0.83}), (4, 5, {'weight': 0.16}), (4, 7, {'weight': 0.88}), (4, 8, {'weight': 0.8}), (4, 9, {'weight': 0.46}), (5, 6, {'weight': 0.75}), (5, 7, {'weight': 0.83}), (5, 8, {'weight': 0.73}), (5, 9, {'weight': 0.02}), (6, 7, {'weight': 0.07}), (6, 8, {'weight': 0.69}), (6,

In [11]:
len(graphs_edgelist_list_dict['er_graph_0'].edges(data=True))

41

The graph after prediction is shifted by 1 unit. For example, in NetworkX the edge is (0, 2, 0.36), but in this DataFrame it becomes (1, 3, 0.36)

In [12]:
qaoa_gpt_circ_df[:1]

graph  \
0  [(1, 2), 0.02, (1, 3), 0.6, (1, 4), 0.88, (1, 5), 0.76, (1, 6), 0.33, (1, 7), 0.02, (1, 8), 0.47, (1, 9), 0.86, (1, 10), 0.97, (2, 3), 0.38, (2, 4), 0.03, (2, 5), 0.71, (2, 7), 0.98, (2, 8), 0.65, (2, 9), 0.82, (2, 10), 0.35, (3, 4), 0.74, (3, 5), 0.42, (3, 6), 0.4, (3, 7), 0.1, (3, 8), 0.97, (3, 9), 0.28, (3, 10), 0.5, (4, 5), 0.53, (4, 6), 0.58, (4, 7), 0.85, (4, 8), 0.84, (4, 10), 0.83, (5, 6), 0.16, (5, 8), 0.88, (5, 9), 0.8, (5, 10), 0.46, (6, 7), 0.75, (6, 8), 0.83, (6, 9), 0.73, (6, 10), 0.02, (7, 8), 0.07, (7, 9), 0.69, (7, 10), 0.71, (8, 9), 0.64, (9, 10), 0.79]   

   n_edges  \
0       41   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [13]:
qaoa_gpt_circ_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   graph              5 non-null      object 
 1   n_edges            5 non-null      int64  
 2   q_circuits         5 non-null      object 
 3   adapt_circuit      5 non-null      object 
 4   adapt_full_ar      0 non-null      object 
 5   graph_prefix       5 non-null      object 
 6   energy_gurobi      5 non-null      float64
 7   label              5 non-null      object 
 8   graph_w_jl         5 non-null      object 
 9   graph_weight_norm  5 non-null      float64
dtypes: float64(2), int64(1), object(7)
memory usage: 528.0+ bytes


## Evaluate circuits

In [14]:
qaoa_gpt_circ_eval_df = qaoa_gpt.eval_circ_df_jl(qaoa_gpt_circ_df)


===== DEBUG INFO =====
CWD: /home/mrzaizai2k/code_bao/ADAPT_GPT
Command:
/opt/julia-1.12.1/bin/julia -t 4 --project=/home/mrzaizai2k/code_bao/ADAPT_GPT/ADAPT.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/adapt_gpt_eval_energy.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-04-09__16_46_46_df.json /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-04-09__16_46_46_df_jl.json 10 qaoa_double_pool


Julia return code: 0


In [15]:
sample_gr

EdgeDataView([(0, 1, {'weight': 0.02}), (0, 2, {'weight': 0.6}), (0, 3, {'weight': 0.88}), (0, 4, {'weight': 0.76}), (0, 5, {'weight': 0.33}), (0, 6, {'weight': 0.02}), (0, 7, {'weight': 0.47}), (0, 8, {'weight': 0.86}), (0, 9, {'weight': 0.97}), (1, 2, {'weight': 0.38}), (1, 3, {'weight': 0.03}), (1, 4, {'weight': 0.71}), (1, 6, {'weight': 0.98}), (1, 7, {'weight': 0.65}), (1, 8, {'weight': 0.82}), (1, 9, {'weight': 0.35}), (2, 3, {'weight': 0.74}), (2, 4, {'weight': 0.42}), (2, 5, {'weight': 0.4}), (2, 6, {'weight': 0.1}), (2, 7, {'weight': 0.97}), (2, 8, {'weight': 0.28}), (2, 9, {'weight': 0.5}), (3, 4, {'weight': 0.53}), (3, 5, {'weight': 0.58}), (3, 6, {'weight': 0.85}), (3, 7, {'weight': 0.84}), (3, 9, {'weight': 0.83}), (4, 5, {'weight': 0.16}), (4, 7, {'weight': 0.88}), (4, 8, {'weight': 0.8}), (4, 9, {'weight': 0.46}), (5, 6, {'weight': 0.75}), (5, 7, {'weight': 0.83}), (5, 8, {'weight': 0.73}), (5, 9, {'weight': 0.02}), (6, 7, {'weight': 0.07}), (6, 8, {'weight': 0.69}), (6,

The eval df add 1 more col is adapt_gpt_energies

Each graph is generate into 5 other graphs, that why we see list of 5 q_circuits and adapt_gpt_energies



In [16]:
qaoa_gpt_circ_eval_df[:1]

graph_prefix  \
0   er_graph_0   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    graph  \
0  [[1, 2], 0.02, [1, 3], 0.6000000000000001, [1, 4], 0.88, [1, 5], 0.76, [1, 6], 0.33, [1, 7], 0.02, [1, 8], 0.47000000000000003, [1, 9], 0.86, [1, 10], 0.97, [2, 3], 0.38, [2, 4], 0.03, [2, 5], 0.71, [2, 7], 0.98, [2, 8], 0.65, [2, 9], 0.8200000000000001, [2, 10], 0.35000000000000003, [3, 4], 0.74, [3, 5], 0.42, [3, 6], 0.4, [3, 7], 0.1, [3, 8], 0.97, [3, 9], 0.28, [3, 10], 0.5, [4, 5], 0.53, [4, 6], 0.58, [4, 7], 0.85, [4, 8], 0.84, [4, 10], 0.8300000000000001, [5, 6], 0.16, [5, 8], 0.88, [5, 9], 0.8, [5, 10], 0.46, [6, 7], 0.75, [6, 8], 0.8300000000000001, [6, 9], 0.73, [6, 10], 0.02, [7, 8], 0.07, [7, 9], 0.6900000000000001, [7, 10], 0.71, [8, 9], 0.64, [9, 10], 0.79]   

   n_edges  \
0       41   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [17]:
qaoa_gpt_circ_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   graph_prefix        5 non-null      object 
 1   graph               5 non-null      object 
 2   n_edges             5 non-null      int64  
 3   q_circuits          5 non-null      object 
 4   adapt_gpt_energies  5 non-null      object 
 5   energy_gurobi       5 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 368.0+ bytes


In [18]:
# 3 extract first rows into 5 rows for 5 q_circuits and adapt_gpt_energies
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits']) 

In [19]:
qaoa_gpt_circ_eval_expl_df[:2]


,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 2], 0.02, [1, 3], 0.6000000000000001, [1, 4], 0.88, [1, 5], 0.76, [1, 6], 0.33, [1, 7], 0.02, [1, 8], 0.47000000000000003, [1, 9], 0.86, [1, 10], 0.97, [2, 3], 0.38, [2, 4], 0.03, [2, 5], 0.71, [2, 7], 0.98, [2, 8], 0.65, [2, 9], 0.8200000000000001, [2, 10], 0.35000000000000003, [3, 4], 0.74, [3, 5], 0.42, [3, 6], 0.4, [3, 7], 0.1, [3, 8], 0.97, [3, 9], 0.28, [3, 10], 0.5, [4, 5], 0.53, [4, 6], 0.58, [4, 7], 0.85, [4, 8], 0.84, [4, 10], 0.8300000000000001, [5, 6], 0.16, [5, 8], 0.88, [5, 9], 0.8, [5, 10], 0.46, [6, 7], 0.75, [6, 8], 0.8300000000000001, [6, 9], 0.73, [6, 10], 0.02, [7, 8], 0.07, [7, 9], 0.6900000000000001, [7, 10], 0.71, [8, 9], 0.64, [9, 10], 0.79]",41,"[new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, [7, 10], 0.29, 0.29, 0.29, 0.29, ...]",999,-15.35
0,er_graph_0,"[[1, 2], 0.02, [1, 3], 0.6000000000000001, [1, 4], 0.88, [1, 5], 0.76, [1, 6], 0.33, [1, 7], 0.02, [1, 8], 0.47000000000000003, [1, 9], 0.86, [1, 10], 0.97, [2, 3], 0.38, [2, 4], 0.03, [2, 5], 0.71, [2, 7], 0.98, [2, 8], 0.65, [2, 9], 0.8200000000000001, [2, 10], 0.35000000000000003, [3, 4], 0.74, [3, 5], 0.42, [3, 6], 0.4, [3, 7], 0.1, [3, 8], 0.97, [3, 9], 0.28, [3, 10], 0.5, [4, 5], 0.53, [4, 6], 0.58, [4, 7], 0.85, [4, 8], 0.84, [4, 10], 0.8300000000000001, [5, 6], 0.16, [5, 8], 0.88, [5, 9], 0.8, [5, 10], 0.46, [6, 7], 0.75, [6, 8], 0.8300000000000001, [6, 9], 0.73, [6, 10], 0.02, [7, 8], 0.07, [7, 9], 0.6900000000000001, [7, 10], 0.71, [8, 9], 0.64, [9, 10], 0.79]",41,"[new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, [8, 10], 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, [2, 7], 0.29, [7, 10], 0.26, [3, 7], 0.29, [2, 7], 0.26, [2, 7], 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, ...]",999,-15.35


In [20]:
qaoa_gpt_circ_eval_expl_df[qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] > 0][:2]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 2], 0.02, [1, 3], 0.6000000000000001, [1, 4], 0.88, [1, 5], 0.76, [1, 6], 0.33, [1, 7], 0.02, [1, 8], 0.47000000000000003, [1, 9], 0.86, [1, 10], 0.97, [2, 3], 0.38, [2, 4], 0.03, [2, 5], 0.71, [2, 7], 0.98, [2, 8], 0.65, [2, 9], 0.8200000000000001, [2, 10], 0.35000000000000003, [3, 4], 0.74, [3, 5], 0.42, [3, 6], 0.4, [3, 7], 0.1, [3, 8], 0.97, [3, 9], 0.28, [3, 10], 0.5, [4, 5], 0.53, [4, 6], 0.58, [4, 7], 0.85, [4, 8], 0.84, [4, 10], 0.8300000000000001, [5, 6], 0.16, [5, 8], 0.88, [5, 9], 0.8, [5, 10], 0.46, [6, 7], 0.75, [6, 8], 0.8300000000000001, [6, 9], 0.73, [6, 10], 0.02, [7, 8], 0.07, [7, 9], 0.6900000000000001, [7, 10], 0.71, [8, 9], 0.64, [9, 10], 0.79]",41,"[new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, [7, 10], 0.29, 0.29, 0.29, 0.29, ...]",999,-15.35
0,er_graph_0,"[[1, 2], 0.02, [1, 3], 0.6000000000000001, [1, 4], 0.88, [1, 5], 0.76, [1, 6], 0.33, [1, 7], 0.02, [1, 8], 0.47000000000000003, [1, 9], 0.86, [1, 10], 0.97, [2, 3], 0.38, [2, 4], 0.03, [2, 5], 0.71, [2, 7], 0.98, [2, 8], 0.65, [2, 9], 0.8200000000000001, [2, 10], 0.35000000000000003, [3, 4], 0.74, [3, 5], 0.42, [3, 6], 0.4, [3, 7], 0.1, [3, 8], 0.97, [3, 9], 0.28, [3, 10], 0.5, [4, 5], 0.53, [4, 6], 0.58, [4, 7], 0.85, [4, 8], 0.84, [4, 10], 0.8300000000000001, [5, 6], 0.16, [5, 8], 0.88, [5, 9], 0.8, [5, 10], 0.46, [6, 7], 0.75, [6, 8], 0.8300000000000001, [6, 9], 0.73, [6, 10], 0.02, [7, 8], 0.07, [7, 9], 0.6900000000000001, [7, 10], 0.71, [8, 9], 0.64, [9, 10], 0.79]",41,"[new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, new_layer_p, [8, 10], 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, [2, 7], 0.29, [7, 10], 0.26, [3, 7], 0.29, [2, 7], 0.26, [2, 7], 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, 0.29, ...]",999,-15.35


This computes how close each QAOA-GPT circuit’s energy is to the optimal solution (AR — approximation ratio).

If the ratio = 1 → perfect solution.

If <1 → circuit energy is above the optimal energy (since MaxCut is a minimization problem in some conventions, sometimes they flip it; conceptually, ratio shows performance).

In [21]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(-41.67645996496824)

In [22]:
# on avg, how many layers are there in the predicted circuits
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(55.28)

The code above is incorrect becasue it not count the error circuits which have adapt_gpt_energies = 999, so you shoul used compute_metrics

In [23]:
ar, err, layers = compute_metrics(qaoa_gpt_circ_eval_df)
print(f"Avg AR: {ar}, Avg ERR: {err}, Avg Layers: {layers}")

Avg AR: 0.9519244132289235, Avg ERR: 0.6, Avg Layers: 55.28000000000001
